# Clonación de voz con XTTS-v2


**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU T4**.

Reglas para que todo funcione:
1. SER RESPONSABLE, NO USAR PARA COSAS ILEGALES.


## Parte 0: Instalación

In [ ]:
!nvidia-smi | head -n 12

Thu Oct  1 00:35:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Todo en un solo comando, con las versiones que son compatibles entre sí.
# No uses --force-reinstall.
!pip install -q coqui-tts "transformers>=4.47,<5" "numpy<2.3" "requests==2.32.4" "fsspec==2025.12.0" faster-whisper pydub soundfile num2words
!apt-get -qq install -y ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.6/39.6 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 639.3/639.3 kB 23.1 MB/s eta 0:00:00
  

### Ahora reinicia la sesión
**Entorno de ejecución → Reiniciar sesión.** Luego ejecuta la celda de verificación (no repitas la instalación).

In [ ]:
import traceback
try:
    import transformers, torch
    from TTS.tts.configs.xtts_config import XttsConfig
    from TTS.tts.models.xtts import Xtts, XttsAudioConfig
    from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig
    from trainer import Trainer, TrainerArgs
    print("transformers", transformers.__version__, "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())
    print("TODO OK: puedes continuar")
except Exception:
    traceback.print_exc()
    print("\nHAY UN ERROR: copia el mensaje completo (desde la primera línea).")

transformers 4.57.6 | torch 2.11.0+cu128 | GPU: True
TODO OK: puedes continuar


## Parte 1: Subir y preparar tu audio
Si ya subiste el archivo al panel izquierdo de Colab, escribe su nombre en `NOMBRE_ARCHIVO`. Si no, déjalo en `None` y te pedirá subirlo.

In [ ]:
import os
from pydub import AudioSegment, effects

NOMBRE_ARCHIVO = None   # ej: "mi_audio.m4a"  (None = subirlo ahora)

if NOMBRE_ARCHIVO is None:
    from google.colab import files
    subido = files.upload()
    NOMBRE_ARCHIVO = list(subido.keys())[0]

audio = AudioSegment.from_file(NOMBRE_ARCHIVO)
audio = audio.set_channels(1).set_frame_rate(22050)
audio = effects.normalize(audio)
audio.export("audio_limpio.wav", format="wav")
print(NOMBRE_ARCHIVO, f"-> {len(audio)/60000:.1f} minutos")

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Saving Audio.mp4 to Audio.mp4
Audio.mp4 -> 15.5 minutos


## Parte 2: Transcribir y cortar en clips
Whisper transcribe con tiempos por palabra y se cortan clips de 3 a 10.5 segundos. Se descartan solos los textos repetitivos y los créditos inventados ("Get Images", etc.). Los números se convierten a letras.

Esta celda **borra y recrea la carpeta `dataset/`**. Si ya tienes correcciones hechas, no la repitas.

In [ ]:
import os, re, gc, numpy as np, torch
from pydub import AudioSegment
from faster_whisper import WhisperModel

audio = AudioSegment.from_wav("audio_limpio.wav")
a16 = audio.set_frame_rate(16000).set_channels(1)
arr = np.array(a16.get_array_of_samples()).astype(np.float32) / 32768.0   # sin usar PyAV

whisper = WhisperModel("large-v3", device="cuda", compute_type="float16")
segmentos, _ = whisper.transcribe(
    arr, language="es", vad_filter=True, beam_size=5,
    word_timestamps=True, condition_on_previous_text=False,   # evita repeticiones
)
palabras = [w for s in segmentos for w in (s.words or [])]
del whisper
gc.collect(); torch.cuda.empty_cache()
print(len(palabras), "palabras transcritas")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocabulary.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

1645 palabras transcritas


In [ ]:
import shutil
from num2words import num2words

MIN_S, MAX_S = 3.0, 10.5
MALAS = ["get images", "getty", "amara.org", "subtítulos", "suscríbete", "gracias por ver"]

if os.path.exists("dataset"):
    shutil.rmtree("dataset")
os.makedirs("dataset/wavs")

def limpiar(t):
    t = re.sub(r"\s+", " ", t).strip()
    t = re.sub(r"\d+", lambda m: num2words(int(m.group()), lang="es"), t)
    return t.replace("|", " ")

def repetitivo(t):
    p = t.lower().split()
    return len(p) >= 6 and len(set(p)) / len(p) < 0.5

clips = []
def cerrar(ws):
    if ws[-1].end - ws[0].start >= MIN_S:
        clips.append((ws[0].start, ws[-1].end, "".join(w.word for w in ws)))

cur = []
for i, w in enumerate(palabras):
    if cur and (w.end - cur[0].start) > MAX_S:
        cerrar(cur); cur = []
    cur.append(w)
    hueco = palabras[i + 1].start - w.end if i + 1 < len(palabras) else 99
    dur = w.end - cur[0].start
    fin_frase = w.word.strip()[-1:] in ".?!"
    if dur >= MIN_S and (hueco > 0.5 or (fin_frase and dur >= 5)):
        cerrar(cur); cur = []
if cur:
    cerrar(cur)

meta, descartados = {}, []
for i, (a, b, t) in enumerate(clips):
    t = limpiar(t)
    if not t or repetitivo(t) or any(m in t.lower() for m in MALAS):
        descartados.append((i, t)); continue
    nombre = f"clip_{i:04d}"
    audio[max(0, int((a - 0.05) * 1000)): int((b + 0.15) * 1000)].export(f"dataset/wavs/{nombre}.wav", format="wav")
    meta[nombre] = t

with open("dataset/metadata.csv", "w", encoding="utf-8") as f:
    for k, t in meta.items():
        f.write(f"{k}|{t}|{t}\n")

print(len(meta), "clips guardados;", len(descartados), "descartados automáticamente")
for i, t in descartados:
    print("  descartado:", t[:70])

101 clips guardados; 1 descartados automáticamente
  descartado: Fuente, Arslan Hider, Get Images.


## Parte 3: Revisar y corregir el texto (sin Excel)
Lee cada línea, escucha los clips dudosos y corrige. **Cada clip debe decir exactamente lo que se oye.** Si no estás seguro de un clip, bórralo.

In [ ]:
import os
from IPython.display import Audio, display
from pydub import AudioSegment

RUTA = "dataset/metadata.csv"

def leer_meta():
    d = {}
    with open(RUTA, encoding="utf-8") as f:
        for l in f:
            l = l.strip()
            if l:
                c = l.split("|")
                d[c[0]] = c[1] if len(c) > 1 else ""
    return d

def guardar_meta():
    with open(RUTA, "w", encoding="utf-8") as f:
        for k, t in meta.items():
            t = t.replace("|", " ").strip()
            if t:
                f.write(f"{k}|{t}|{t}\n")

def dur(k):
    return len(AudioSegment.from_wav(f"dataset/wavs/{k}.wav")) / 1000

def escuchar(k):
    print(k, f"[{dur(k):.1f}s]", meta[k])
    display(Audio(f"dataset/wavs/{k}.wav"))

def corregir(k, texto_nuevo):
    meta[k] = texto_nuevo
    guardar_meta()
    print("Corregido:", k)

def borrar(*claves):
    for k in claves:
        meta.pop(k, None)
    guardar_meta()
    print("Borrados:", claves, "| quedan", len(meta))

def verificar():
    lineas = [l for l in open(RUTA, encoding="utf-8").read().split("\n") if l.strip()]
    problemas = 0
    for l in lineas:
        c = l.split("|")
        if len(c) != 3 or not c[2].strip() or not os.path.exists(f"dataset/wavs/{c[0]}.wav"):
            print("PROBLEMA:", l[:80]); problemas += 1
        elif any(ch.isdigit() for ch in c[2]):
            print("Aviso, tiene números (escríbelos con letras):", c[0])
    total = sum(dur(l.split("|")[0]) for l in lineas if os.path.exists(f"dataset/wavs/{l.split('|')[0]}.wav"))
    print(f"{len(lineas)} filas, {problemas} problemas, {total/60:.1f} minutos de audio")

meta = leer_meta()
for k, t in meta.items():
    print(f"{k} [{dur(k):4.1f}s] {t}")

clip_0000 [ 5.0s] Hola, muy buenas tardes a todos. Hola, muy buen día a todos.
clip_0001 [ 9.7s] Espero que se encuentren muy bien. Mi nombre es Jolman Mesa, soy ingeniero electrónico de la Universidad Nacional de Colombia, sede de Manizales.
clip_0002 [ 6.4s] Hoy les quiero hablar sobre inteligencia artificial, bases de datos y visión por computadora.
clip_0003 [ 5.5s] computadora. La inteligencia
clip_0004 [10.6s] artificial, conocida como IA, las bases de datos modernas y la visión por computadora han creado uno de los ecosistemas tecnológicos más poderosos de nuestra era.
clip_0005 [10.6s] Ya no operan como disciplinas aisladas, juntas forman una arquitectura cognitiva completa donde la visión por computadora actúa como los ojos, las bases de datos
clip_0006 [ 3.6s] como la memoria y la inteligencia artificial como el cerebro.
clip_0007 [10.7s] Para comprender el impacto de esta integración, es fundamental desglosar cómo funciona cada pilar y cómo se entrelazan para resolver proble

In [ ]:
borrar("clip_0045")

Borrados: ('clip_0045',) | quedan 100


**Cómo corregir:** usa estas funciones en una celda nueva (se guardan solas).

```python
escuchar("clip_0005")                       # oír el clip y ver su texto
corregir("clip_0005", "Texto correcto.")    # cambiar el texto
borrar("clip_0017", "clip_0035")            # eliminar clips malos
verificar()                                 # comprobar todo antes de entrenar
```

Qué borrar o corregir: créditos o texto que no dijiste, frases duplicadas que no pronunciaste, cortes a media palabra, palabras mal transcritas y siglas (escribe como se pronuncia: "ese cu ele").

In [ ]:
verificar()

100 filas, 0 problemas, 13.7 minutos de audio


In [ ]:
# Copia de seguridad en Google Drive (recomendado)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/mi_voz_proyecto
!cp -r dataset audio_limpio.wav /content/drive/MyDrive/mi_voz_proyecto/
print("Dataset guardado en Drive")

## Parte 4: Modelo base + prueba rápida (zero-shot)
Si algo falla con `torchcodec` en este paso o en el entrenamiento, ejecuta `!pip install -q torchcodec` y repite la celda.

In [ ]:
import os
from huggingface_hub import hf_hub_download

BASE = os.path.abspath("xtts_base")
for f in ["model.pth", "config.json", "vocab.json", "dvae.pth", "mel_stats.pth"]:
    hf_hub_download("coqui/XTTS-v2", f, local_dir=BASE)
print(os.listdir(BASE))

model.pth:   0%|          | 0.00/1.87G [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

dvae.pth:   0%|          | 0.00/211M [00:00<?, ?B/s]

mel_stats.pth:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

['model.pth', 'mel_stats.pth', 'config.json', 'vocab.json', '.cache', 'dvae.pth']


**Funciones de síntesis.** Si reinicias la sesión, vuelve a ejecutar esta celda antes de generar audio.

In [ ]:
import re, numpy as np, soundfile as sf, torch
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

def cargar_modelo(config_path, checkpoint_path, vocab_path):
    cfg = XttsConfig()
    cfg.load_json(config_path)
    m = Xtts.init_from_config(cfg)
    m.load_checkpoint(cfg, checkpoint_path=checkpoint_path,
                      vocab_path=vocab_path, use_deepspeed=False)
    m.cuda()
    m.eval()
    return m          # importante: devolver el modelo

def dividir_texto(texto, max_chars=220):
    texto = re.sub(r"\s+", " ", texto.replace("\n", " ")).strip()
    frases = re.split(r"(?<=[.!?;:])\s+", texto)
    trozos, actual = [], ""
    for f in frases:
        partes = re.split(r"(?<=,)\s+", f) if len(f) > max_chars else [f]
        for p in partes:
            while len(p) > max_chars:                      # sin signos: cortar por palabras
                corte = p.rfind(" ", 0, max_chars)
                corte = corte if corte > 0 else max_chars
                if actual:
                    trozos.append(actual); actual = ""
                trozos.append(p[:corte].strip()); p = p[corte:].strip()
            if len(actual) + len(p) + 1 <= max_chars:
                actual = (actual + " " + p).strip()
            else:
                if actual:
                    trozos.append(actual)
                actual = p
    if actual:
        trozos.append(actual)
    return [t for t in trozos if t]

def recortar(w, umbral=0.01, margen=int(24000 * 0.05)):
    idx = np.where(np.abs(w) > umbral)[0]
    if len(idx) == 0:
        return w
    return w[max(0, idx[0] - margen): idx[-1] + margen]

@torch.inference_mode()
def sintetizar(modelo, texto, voz_ref, salida="salida.wav",
               temperatura=0.65, velocidad=1, pausa_s=0.10, ganancia=2.0):
    latente, embedding = modelo.get_conditioning_latents(
        audio_path=[voz_ref],
        gpt_cond_len=modelo.config.gpt_cond_len,
        max_ref_length=modelo.config.max_ref_len,
        sound_norm_refs=modelo.config.sound_norm_refs,
    )
    trozos = dividir_texto(texto)
    pausa = np.zeros(int(24000 * pausa_s), dtype=np.float32)
    piezas = []
    for i, t in enumerate(trozos, 1):
        print(f"[{i}/{len(trozos)}] {t[:60]}")
        out = modelo.inference(
            t, "es", latente, embedding,
            temperature=temperatura, repetition_penalty=5.0,
            top_k=50, top_p=0.85, speed=velocidad,
            enable_text_splitting=False,
        )
        w = out["wav"]
        if hasattr(w, "cpu"):
            w = w.cpu().numpy()
        w = recortar(np.asarray(w, dtype=np.float32).squeeze())
        piezas += [w, pausa]
    final = np.concatenate(piezas)
    final = final / (np.max(np.abs(final)) + 1e-8)               # normalizar al máximo
    final = np.tanh(final * ganancia) / np.tanh(ganancia)         # sube el volumen percibido
    sf.write(salida, final * 0.95, 24000)
    return salida

In [ ]:
# Audio de referencia automático: ~20 s armados con varios de tus clips
import glob, random
from pydub import AudioSegment

random.seed(1)
wavs = glob.glob("dataset/wavs/*.wav")
random.shuffle(wavs)
ref = AudioSegment.silent(0)
for p in wavs:
    c = AudioSegment.from_wav(p)
    if 5000 <= len(c) <= 11000:
        ref += c + AudioSegment.silent(200)
    if len(ref) >= 20000:
        break
ref.export("referencia.wav", format="wav")
print(f"referencia.wav: {len(ref)/1000:.1f} s")

referencia.wav: 21.5 s


In [ ]:
base = cargar_modelo(f"{BASE}/config.json", f"{BASE}/model.pth", f"{BASE}/vocab.json")
print(type(base))     # debe decir ...Xtts'>

texto = """Hola, esta es una prueba de mi voz sintética. Puedo escribir textos muy largos
y el sistema los dividirá en frases para generar el audio completo."""

sintetizar(base, texto, "referencia.wav", "prueba_zeroshot.wav")

from IPython.display import Audio
Audio("prueba_zeroshot.wav")

<class 'TTS.tts.models.xtts.Xtts'>
[1/1] Hola, esta es una prueba de mi voz sintética. Puedo escribir


Si ya suena como quieres, puedes saltar a la Parte 7. Si quieres más parecido, continúa con el fine-tuning.

## Parte 5: Fine-tuning

Con pocos datos, el número de **pasos de optimización** importa mucho. Esta configuración usa `batch_size=2` y `grad_accum_steps=2`, para que el modelo se actualice varias veces por época. Con 15 a 20 minutos de audio, 10 épocas es un buen punto de partida.

**Recomendado:** reinicia la sesión antes de entrenar para liberar memoria de GPU y repite la celda de funciones y la del modelo base solo si te las pide.

In [ ]:
import os, gc, torch
from trainer import Trainer, TrainerArgs
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig
from TTS.tts.models.xtts import XttsAudioConfig

for n in ["base", "mi_modelo", "whisper"]:      # liberar GPU
    globals().pop(n, None)
gc.collect(); torch.cuda.empty_cache()

BASE = os.path.abspath("xtts_base")
DATA = os.path.abspath("dataset")
OUT_PATH = os.path.abspath("entrenamiento")
os.makedirs(OUT_PATH, exist_ok=True)

EPOCHS = 10
BATCH = 2
ACUM = 2           # si hay error de memoria: BATCH=1 y ACUM=4

dataset = BaseDatasetConfig(
    formatter="ljspeech",
    dataset_name="mi_voz",
    path=DATA,
    meta_file_train="metadata.csv",      # solo el nombre del archivo
    language="es",
)

model_args = GPTArgs(
    max_conditioning_length=132300,
    min_conditioning_length=66150,
    debug_loading_failures=False,
    max_wav_length=255995,
    max_text_length=200,
    mel_norm_file=f"{BASE}/mel_stats.pth",
    dvae_checkpoint=f"{BASE}/dvae.pth",
    xtts_checkpoint=f"{BASE}/model.pth",
    tokenizer_file=f"{BASE}/vocab.json",
    gpt_num_audio_tokens=1026,
    gpt_start_audio_token=1024,
    gpt_stop_audio_token=1025,
    gpt_use_masking_gt_prompt_approach=True,
    gpt_use_perceiver_resampler=True,
)

audio_config = XttsAudioConfig(sample_rate=22050, dvae_sample_rate=22050, output_sample_rate=24000)

config = GPTTrainerConfig(
    epochs=EPOCHS,
    output_path=OUT_PATH,
    model_args=model_args,
    run_name="mi_voz_xtts",
    project_name="clon_voz",
    dashboard_logger="tensorboard",
    audio=audio_config,
    batch_size=BATCH,
    batch_group_size=48,
    eval_batch_size=BATCH,
    num_loader_workers=2,
    eval_split_max_size=256,
    print_step=10,
    plot_step=100,
    log_model_step=1000,
    save_step=50,
    save_n_checkpoints=1,
    save_checkpoints=True,
    optimizer="AdamW",
    optimizer_wd_only_on_weights=True,
    optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
    lr=5e-6,
    lr_scheduler="MultiStepLR",
    lr_scheduler_params={"milestones": [900000, 2700000, 5400000], "gamma": 0.5, "last_epoch": -1},
    test_sentences=[],
)

model = GPTTrainer.init_from_config(config)

train_samples, eval_samples = load_tts_samples(
    [dataset], eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=0.15,                 # con pocos clips, el valor por defecto daría 0
)
print(len(train_samples), "para entrenar,", len(eval_samples), "para evaluar")

trainer = Trainer(
    TrainerArgs(restore_path=None, skip_train_epoch=False,
                start_with_eval=False, grad_accum_steps=ACUM),
    config,
    output_path=OUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)
trainer.fit()

 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: False
 | > Precision: float32
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 2
 | > Num. of Torch Threads: 2
 | > Torch seed: 1
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False


85 para entrenar, 15 para evaluar


 > Start Tensorboard: tensorboard --logdir=/content/entrenamiento/mi_voz_xtts-October-01-2026_12+51AM-0000000

 > Model has 518442047 parameters

 > EPOCH: 0/9
 --> /content/entrenamiento/mi_voz_xtts-October-01-2026_12+51AM-0000000

 > TRAINING (2026-10-01 00:51:58) 

   --> TIME: 2026-10-01 00:52:00 -- STEP: 0/42 -- GLOBAL_STEP: 0
     | > current_lr: 5e-06  (5e-06)
     | > loss_text_ce: 0.02593637816607952  (0.02593637816607952)
     | > loss_mel_ce: 3.8792600631713867  (3.8792600631713867)
     | > loss: 1.952598214149475  (1.952598214149475)
     | > step_time: 0.9582  (0.9582133293151855)
     | > loader_time: 0.7161  (0.7161459922790527)


   --> TIME: 2026-10-01 00:52:08 -- STEP: 10/42 -- GLOBAL_STEP: 10
     | > current_lr: 5e-06  (5e-06)
     | > loss_text_ce: 0.02424590103328228  (0.025877713039517403)
     | > loss_mel_ce: 4.0900373458862305  (3.912976837158203)
     | > loss: 2.0571415424346924  (1.9694272875785828)
     | > step_time: 0.3323  (0.4768959999084473)
     | >

## Parte 6: Usar tu modelo entrenado
Se busca el último entrenamiento, se quita lo que sobra del checkpoint (optimizador y dvae) y se carga con la configuración del modelo base.

In [ ]:
import glob, os, torch

BASE = os.path.abspath("xtts_base")
run_dir = sorted(glob.glob("entrenamiento/mi_voz_xtts*"), key=os.path.getmtime)[-1]
print("Entrenamiento:", run_dir)

best = f"{run_dir}/best_model.pth"
if os.path.exists(best):
    ckpt = best
else:
    ckpt = sorted(glob.glob(f"{run_dir}/checkpoint_*.pth"), key=os.path.getmtime)[-1]
print("Usando:", ckpt)

ck = torch.load(ckpt, map_location="cpu", weights_only=False)
ck.pop("optimizer", None)
ck.pop("scaler", None)
sd = ck.get("model", ck)
for k in list(sd.keys()):
    if "dvae" in k:
        del sd[k]
torch.save(ck, "modelo_final.pth")
del ck, sd
print("modelo_final.pth listo")

Entrenamiento: entrenamiento/mi_voz_xtts-October-01-2026_12+51AM-0000000
Usando: entrenamiento/mi_voz_xtts-October-01-2026_12+51AM-0000000/best_model.pth
modelo_final.pth listo


In [ ]:
# (Si reiniciaste la sesión, ejecuta antes la celda de "Funciones de síntesis" de la Parte 4)
mi_modelo = cargar_modelo(f"{BASE}/config.json", "modelo_final.pth", f"{BASE}/vocab.json")
print(type(mi_modelo))

texto = """ La tecnología ha cambiado profundamente la manera en que las personas trabajan, estudian, se comunican y toman decisiones. En los últimos años, uno de los campos que más ha crecido es la inteligencia artificial, una disciplina que busca desarrollar sistemas capaces de realizar tareas que normalmente requieren capacidades humanas, como reconocer patrones, comprender información, analizar datos, interpretar imágenes y generar contenido.

La inteligencia artificial no es solamente un conjunto de programas modernos. Es un campo amplio que combina matemáticas, estadística, programación, procesamiento de datos y aprendizaje automático. Gracias a estas tecnologías, una computadora puede aprender a partir de ejemplos y utilizar lo aprendido para realizar predicciones o tomar determinadas decisiones. """

sintetizar(mi_modelo, texto, "referencia.wav", "prueba_finetuned.wav")

from IPython.display import Audio
Audio("prueba_finetuned.wav")

<class 'TTS.tts.models.xtts.Xtts'>
[1/5] La tecnología ha cambiado profundamente la manera en que las
[2/5] una disciplina que busca desarrollar sistemas capaces de rea
[3/5] interpretar imágenes y generar contenido. La inteligencia ar
[4/5] Es un campo amplio que combina matemáticas, estadística, pro
[5/5] Gracias a estas tecnologías, una computadora puede aprender 


In [ ]:
# Guardar el modelo en Drive para no perderlo
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/mi_voz_modelo
!cp modelo_final.pth referencia.wav /content/drive/MyDrive/mi_voz_modelo/
!cp xtts_base/config.json xtts_base/vocab.json /content/drive/MyDrive/mi_voz_modelo/
print("Modelo guardado en Drive/mi_voz_modelo")

Mounted at /content/drive
Modelo guardado en Drive/mi_voz_modelo


## Parte 7: Generar audio de textos largos
Pega tu texto y ejecuta. Si usas el modelo entrenado, `modelo = mi_modelo`; si prefieres el zero-shot, `modelo = base`.

Ajustes de `temperatura`: baja a 0.5 si inventa sílabas o se repite; sube a 0.75 si suena monótono.

In [ ]:
modelo = mi_modelo       # o: base

TEXTO_LARGO = """
¿Y si esta voz no fuera mía? Pues sí lo es, pero la hizo una inteligencia artificial.

Grabé veinte minutos de mi voz, nada más. Con eso entrené un modelo de síntesis de voz.

Primero, transcribí y corté el audio en clips pequeños. Después, hice el entrenamiento, o fine tuning, sobre un modelo ya existente, y ahora puedo escribir cualquier texto y escucharlo con mi propia voz.

Todo esto lo hice con Python y una GPU gratuita de Google Colab. Costo del proyecto, cero pesos.

¿Quieres que te enseñe cómo se hace? Déjame tu comentario.
"""

sintetizar(modelo, TEXTO_LARGO, "referencia.wav", "resultado.wav", temperatura=0.75)

from IPython.display import Audio
display(Audio("resultado.wav"))

from google.colab import files
files.download("resultado.wav")

[1/3] ¿Y si esta voz no fuera mía? Pues sí lo es, pero la hizo una
[2/3] Primero, transcribí y corté el audio en clips pequeños. Desp
[3/3] Todo esto lo hice con Python y una GPU gratuita de Google Co


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Usar tu modelo en una sesión nueva (sin volver a entrenar)
1. Ejecuta la Parte 0 completa y reinicia.
2. Monta Drive y copia los archivos:
```python
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p modelo && cp /content/drive/MyDrive/mi_voz_modelo/* modelo/
```
3. Ejecuta la celda "Funciones de síntesis" (Parte 4) y:
```python
mi_modelo = cargar_modelo("modelo/config.json", "modelo/modelo_final.pth", "modelo/vocab.json")
modelo = mi_modelo
```
4. Ejecuta la Parte 7 usando `"modelo/referencia.wav"` como referencia.

---
**Licencia:** XTTS-v2 es de uso no comercial. **Ética:** usa solo tu voz o la de personas que lo autoricen.

**Si algo falla:** copia el error completo (desde la primera línea) y dime en qué celda ocurrió.